# Diagnosis multiplex hyperedges selection: intra/inter sub-scales

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import time
from pathlib import Path
import pickle
import networkx as nx
from itertools import chain

In [85]:
# get parent directory
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)

import optim_help.utilities as ut

import optim_help.O_Information as o_infoptim
import optim_help.paral_perm_bootstrap as pp_bootopt
import optim_help.BCa_bootstrap as bca_bootopt
import optim_help.mult_sel_minimal as sel

import optim_help.plots.o_info_bootstrap_plots as o_plots
import optim_help.plots.BCa_plots as bca_plots

### Import data

In [3]:
data_path = '../../DATA/EDI_DIAG' # contains csv files for each diagnoses
EDI_diags_paths = [
    data_path + '/' + f
    for f in os.listdir(data_path)
    if f.lower().endswith('.csv') and os.path.isfile(os.path.join(data_path, f))
]

EDI_diags = {}
for path in EDI_diags_paths:
    EDI_diags[Path(path).stem] = ut.preproc_df(path)
    
print(EDI_diags.keys())

dict_keys(['ANBP', 'ANR', 'BED_OSFED', 'BN'])


In [4]:
EDI_diags['ANBP'].describe(include='all')

,1,2,3,4,5,6,7,8,9,10,...,82,83,84,85,86,87,88,89,90,91
count,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,...,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000,259.000000
mean,2.683398,2.598456,1.474903,1.366795,1.254826,1.135135,2.683398,2.073359,2.447876,2.223938,...,1.243243,1.791506,2.274131,2.003861,2.030888,2.054054,1.046332,0.984556,0.911197,2.559846
std,1.291052,1.403538,1.460841,1.381150,1.376972,1.386913,1.452128,1.480632,1.606728,1.387949,...,1.317261,1.376352,1.394253,1.339611,1.465036,1.262524,1.190147,1.181147,1.246635,1.200488
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,1.500000,0.000000,0.000000,0.000000,0.000000,2.000000,1.000000,1.000000,1.000000,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,2.000000
50%,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000,3.000000,2.000000,3.000000,2.000000,...,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,0.000000,0.000000,3.000000
75%,4.000000,4.000000,3.000000,3.000000,2.000000,2.000000,4.000000,3.000000,4.000000,4.000000,...,2.000000,3.000000,4.000000,3.000000,3.000000,3.000000,2.000000,2.000000,2.000000,3.000000
max,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000


### Import sub_scales data and get candidate multiplets

In [5]:
sub_scales_path = '../../DATA/EDI_det/sub_scales.csv' 
sub_scales = pd.read_csv(sub_scales_path)[['item', 'scales']]

In [6]:
scales_df_dict = {name: group for name, group in sub_scales.groupby("scales")}
items_by_scale = sub_scales.groupby("scales")["item"].apply(list).to_dict()
del items_by_scale['-']

In [7]:
items_by_scale

{'A': [66, 68, 75, 78, 82, 86, 88],
 'B': [4, 5, 28, 38, 46, 53, 61, 64],
 'BD': [2, 9, 12, 19, 31, 45, 47, 55, 59, 62],
 'DT': [1, 7, 11, 16, 25, 32, 49],
 'ED': [67, 70, 72, 79, 81, 83, 85, 90],
 'IA': [17, 30, 54, 65, 74, 76, 80, 89],
 'ID': [8, 21, 26, 33, 40, 44, 51, 60, 77],
 'II': [15, 23, 34, 57, 69, 73, 87],
 'LSE': [10, 27, 37, 41, 42, 50],
 'MF': [3, 6, 14, 22, 35, 39, 48, 58],
 'P': [13, 29, 36, 43, 52, 63],
 'PA': [18, 20, 24, 56, 84, 91]}

In [8]:
all_sub_scales = items_by_scale.copy()  
all_sub_scales['-'] = [71]

In [9]:
all_sub_scales

{'A': [66, 68, 75, 78, 82, 86, 88],
 'B': [4, 5, 28, 38, 46, 53, 61, 64],
 'BD': [2, 9, 12, 19, 31, 45, 47, 55, 59, 62],
 'DT': [1, 7, 11, 16, 25, 32, 49],
 'ED': [67, 70, 72, 79, 81, 83, 85, 90],
 'IA': [17, 30, 54, 65, 74, 76, 80, 89],
 'ID': [8, 21, 26, 33, 40, 44, 51, 60, 77],
 'II': [15, 23, 34, 57, 69, 73, 87],
 'LSE': [10, 27, 37, 41, 42, 50],
 'MF': [3, 6, 14, 22, 35, 39, 48, 58],
 'P': [13, 29, 36, 43, 52, 63],
 'PA': [18, 20, 24, 56, 84, 91],
 '-': [71]}

#### Use the general graph - not diagnosis based, to check links and select inter scales multiplets.

In [10]:
# GRAPHS -------------------------------------------------
graph_data_path = '../../DATA/network_ALL_article.graphml'

G_all = nx.read_graphml(graph_data_path)
# rename the nodes to have the item index as name
mapping = {name: str(int(name.replace("n", "")) + 1) for name in G_all.nodes()}
G_all = nx.relabel_nodes(G_all, mapping)

#### Get all combination multiplets - order: 3, 4, 5

In [11]:
intra_candidate = ut.all_mults_combinations(items_by_scale) # ALL intra combinations

two_inter_candidate = ut.two_interscale_mults_combinations_G_k(all_sub_scales, G_all)
#order_inter_candidate = ut.order_interscale_mults_combinations_G(all_sub_scales, G_all)
# this takes some minutes

In [12]:
total = 0
for couple, results in two_inter_candidate.items():
    for order, combos in results.items():
        total += len(combos)
print(total)

#for couple, results in order_inter_candidate.items():
#    for order, combos in results.items():
#        total += len(combos)
#print(total)

115722


In [13]:
#inter_candidates = {}
#for order in (3,4,5):
#    two_inter = two_inter_candidate[order].values()
#    order_inter = order_inter_candidate[order].values()
#    inter_candidates[order] = list(chain.from_iterable(two_inter)) + list(chain.from_iterable(order_inter))

#inter_multiplets = ut.multiplets_to_df(inter_candidates)

In [14]:
inter_multiplets = ut.multiplets_to_df(two_inter_candidate)

In [15]:
inter_candidate_F = {}
inter_candidate_omega_F = {}
for diagnosis, data in EDI_diags.items():
    start = time.perf_counter()
    inter_candidate_F[diagnosis], inter_candidate_omega_F[diagnosis] = pp_bootopt.filter_multiplets_by_omega(multiplets_dict = inter_multiplets,
                                                                             X = EDI_diags[diagnosis],
                                                                             threshold=0.1)
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Runtime: 274.523096 s
Runtime: 314.894484 s
Runtime: 237.837107 s
Runtime: 353.171676 s


In [58]:
def get_perc(order):
    if order == 3:
        return 0.10
    if order == 4:
        return 0.05
    else:
        return 0.01
    
inter_candidate_FF = {}
for diag, order_df in inter_candidate_omega_F.items():
    print('Diagnosis:', diag)
    inter_candidate_FF[diag] = {}
    for order, df in order_df.items():
        perc = get_perc(order)
        
        df_pos = df[df["omega"] > 0].reset_index(drop=True)
        df_neg = df[df["omega"] < 0].reset_index(drop=True)
        print(f"Order {order}, Positive: {len(df_pos)}, Negative: {len(df_neg)}")
        
        top_pos = df_pos.head(max(1, int(np.ceil(perc * len(df_pos)))))
        top_neg = df_neg.head(max(1, int(np.ceil(perc * len(df_neg)))))
        print(f"Order {order}, Positive: {len(top_pos)}, Negative: {len(top_neg)}")
        
        inter_candidate_FF[diag][order] = pd.concat([top_pos.drop(columns="omega"),
                               top_neg.drop(columns="omega"),],ignore_index=True,)
        print(f"Order {order}, Total: {len(inter_candidate_FF[diag][order])}")
    print('\n')

Diagnosis: ANBP
Order 3, Positive: 1, Negative: 736
Order 3, Positive: 1, Negative: 74
Order 3, Total: 75
Order 4, Positive: 5, Negative: 15625
Order 4, Positive: 1, Negative: 782
Order 4, Total: 783
Order 5, Positive: 71, Negative: 98329
Order 5, Positive: 1, Negative: 984
Order 5, Total: 985


Diagnosis: ANR
Order 3, Positive: 20, Negative: 279
Order 3, Positive: 2, Negative: 28
Order 3, Total: 30
Order 4, Positive: 55, Negative: 15105
Order 4, Positive: 3, Negative: 756
Order 4, Total: 759
Order 5, Positive: 176, Negative: 97270
Order 5, Positive: 2, Negative: 973
Order 5, Total: 975


Diagnosis: BED_OSFED
Order 3, Positive: 0, Negative: 864
Order 3, Positive: 0, Negative: 87
Order 3, Total: 87
Order 4, Positive: 38, Negative: 15163
Order 4, Positive: 2, Negative: 759
Order 4, Total: 761
Order 5, Positive: 3352, Negative: 90411
Order 5, Positive: 34, Negative: 905
Order 5, Total: 939


Diagnosis: BN
Order 3, Positive: 0, Negative: 319
Order 3, Positive: 0, Negative: 32
Order 3, Tota

In [59]:
inter_candidate_FF['BN'][5]

,node1,node2,node3,node4,node5
0,9,45,55,62,32
1,15,73,87,43,63
2,57,73,87,43,63
3,40,44,77,43,52
4,21,40,44,43,63
...,...,...,...,...,...
981,54,74,15,69,87
982,8,40,51,77,73
983,5,61,33,40,77
984,65,74,76,43,63


In [60]:
for diag in inter_candidate_FF:
    tot = 0
    for order in inter_candidate_FF[diag]:
        tot += len(inter_candidate_FF[diag][order])
        print_ = diag + ', ' + str(order) + ':' + str(len(inter_candidate_FF[diag][order]))
    print(diag + ', TOTAL is', str(tot))

ANBP, TOTAL is 1843
ANR, TOTAL is 1764
BED_OSFED, TOTAL is 1787
BN, TOTAL is 1800


In [66]:
intra_candidate_df = {}
for scale, mult_cand in intra_candidate.items():
    intra_candidate_df[scale] = {k: pd.DataFrame(v, columns=[f"node{i+1}" for i in range(k)]) for k, v in mult_cand.items()}

Convert structure of INTRA by merging DFs of same order:
    intra[scale][order] = DF ----> intra[order] = DF

In [69]:
intra_DFs = {}

for order in (3,4,5):
    intra_DFs[order] = pd.concat(
        [scale_dict[order] for scale_dict in intra_candidate_df.values()],
        ignore_index=True,
    )
    print(len(intra_DFs[order]))

593
766
683


In [70]:
intra_DFs[3].head()

,node1,node2,node3
0,66,68,75
1,66,68,78
2,66,68,82
3,66,68,86
4,66,68,88


#### Merge INTRA + INTER

In [71]:
multiplets_diag = {}
for diagnosis, inter_data in inter_candidate_FF.items():
    multiplets_diag[diagnosis] = {}
    for order, inter in inter_data.items():
        multiplets_diag[diagnosis][order] = pd.concat([intra_DFs[order],
                                      inter_candidate_FF[diagnosis][order]], ignore_index=True,)

In [80]:
multiplets_diag['BN'][3].head()

,node1,node2,node3
0,66,68,75
1,66,68,78
2,66,68,82
3,66,68,86
4,66,68,88


### Compute $\Omega$-information and perform the permutation test to select the multiplets

In [74]:
n_boot_perm = 1000 # before: 400

In [81]:
out_dir = "perm_boot_SCALES"            
os.makedirs(out_dir, exist_ok=True)

perm_boot_diag_scale = {}
for diagnosis in EDI_diags:
    start = time.perf_counter()
    
    perm_boot_diag_scale[diagnosis] = pp_bootopt.perm_bootstrap_parallel(multiplets_dict=multiplets_diag[diagnosis],
                                                                        X=EDI_diags[diagnosis], 
                                                                        n_boot=n_boot_perm)
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")
    
    fname = f"{diagnosis}_perm_scales.pkl"
    fpath = os.path.join(out_dir, fname)
    with open(fpath, "wb") as f:
        pickle.dump(perm_boot_diag_scale[diagnosis], f, protocol=pickle.HIGHEST_PROTOCOL)

Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=3:   0%|0/668 |          | 00:00

  Multiplets | order=3:  25%|167/668 |██▌       | 00:48

  Multiplets | order=3:  50%|334/668 |█████     | 01:33

  Multiplets | order=3:  75%|501/668 |███████▌  | 02:20

  Multiplets | order=3: 100%|668/668 |██████████| 03:07

Families:  33%|███▎      | 1/3 [03:07<06:14, 187.32s/it][A

  Multiplets | order=4:   0%|0/1549 |          | 00:00

  Multiplets | order=4:  25%|388/1549 |██▌       | 02:39

  Multiplets | order=4:  50%|776/1549 |█████     | 05:20

  Multiplets | order=4:  75%|1164/1549 |███████▌  | 08:02

  Multiplets | order=4: 100%|1549/1549 |██████████| 10:40

Families:  67%|██████▋   | 2/3 [13:48<07:34, 454.07s/it] 

  Multiplets | order=5:   0%|0/1668 |          | 00:00

  Multiplets | order=5:  25%|417/1668 |██▌       | 03:45

  Multiplets | order=5:  50%|834/1668 |█████     | 07:28

  Multiplets | order=5:  75%|1251/1668 |███████▌  | 11:11

  Multiplets | order=5: 100%

Runtime: 1723.289139 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=3:   0%|0/623 |          | 00:00

  Multiplets | order=3:  25%|156/623 |██▌       | 00:52

  Multiplets | order=3:  50%|312/623 |█████     | 01:48

  Multiplets | order=3:  75%|468/623 |███████▌  | 02:45

  Multiplets | order=3: 100%|623/623 |██████████| 03:40

Families:  33%|███▎      | 1/3 [03:40<07:21, 220.57s/it][A

  Multiplets | order=4:   0%|0/1525 |          | 00:00

  Multiplets | order=4:  25%|382/1525 |██▌       | 03:08

  Multiplets | order=4:  50%|764/1525 |█████     | 06:20

  Multiplets | order=4:  75%|1146/1525 |███████▌  | 09:32

  Multiplets | order=4: 100%|1525/1525 |██████████| 12:41

Families:  67%|██████▋   | 2/3 [16:22<08:58, 538.81s/it] 

  Multiplets | order=5:   0%|0/1658 |          | 00:00

  Multiplets | order=5:  25%|415/1658 |██▌       | 04:35

  Multiplets | order=5:  50%|830/1658 |█████     | 09:15

  Multiplets | order=5:  75%|1245/1658 |███████▌  | 13:54

  Multiplets | order=5: 100%

Runtime: 2093.638107 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=3:   0%|0/680 |          | 00:00

  Multiplets | order=3:  25%|170/680 |██▌       | 00:41

  Multiplets | order=3:  50%|340/680 |█████     | 01:22

  Multiplets | order=3:  75%|510/680 |███████▌  | 02:05

  Multiplets | order=3: 100%|680/680 |██████████| 02:47

Families:  33%|███▎      | 1/3 [02:47<05:35, 167.52s/it][A

  Multiplets | order=4:   0%|0/1527 |          | 00:00

  Multiplets | order=4:  25%|382/1527 |██▌       | 02:11

  Multiplets | order=4:  50%|764/1527 |█████     | 04:27

  Multiplets | order=4:  75%|1146/1527 |███████▌  | 06:42

  Multiplets | order=4: 100%|1527/1527 |██████████| 08:57

Families:  67%|██████▋   | 2/3 [11:44<06:24, 384.91s/it] 

  Multiplets | order=5:   0%|0/1622 |          | 00:00

  Multiplets | order=5:  25%|406/1622 |██▌       | 03:08

  Multiplets | order=5:  50%|812/1622 |█████     | 06:13

  Multiplets | order=5:  75%|1218/1622 |███████▌  | 09:20

  Multiplets | order=5: 100%

Runtime: 1459.213836 s


Families:   0%|          | 0/3 [00:00<?, ?it/s]

  Multiplets | order=3:   0%|0/625 |          | 00:00

  Multiplets | order=3:  25%|157/625 |██▌       | 01:01

  Multiplets | order=3:  50%|314/625 |█████     | 02:02

  Multiplets | order=3:  75%|471/625 |███████▌  | 03:04

  Multiplets | order=3: 100%|625/625 |██████████| 04:04

Families:  33%|███▎      | 1/3 [04:04<08:08, 244.49s/it][A

  Multiplets | order=4:   0%|0/1548 |          | 00:00

  Multiplets | order=4:  25%|387/1548 |██▌       | 03:45

  Multiplets | order=4:  50%|774/1548 |█████     | 07:30

  Multiplets | order=4:  75%|1161/1548 |███████▌  | 11:17

  Multiplets | order=4: 100%|1548/1548 |██████████| 15:04

Families:  67%|██████▋   | 2/3 [19:08<10:32, 632.51s/it] 

  Multiplets | order=5:   0%|0/1669 |          | 00:00

  Multiplets | order=5:  25%|418/1669 |██▌       | 05:26

  Multiplets | order=5:  50%|836/1669 |█████     | 10:46

  Multiplets | order=5:  75%|1254/1669 |███████▌  | 16:08

  Multiplets | order=5: 100%

Runtime: 2438.357294 s


Load pickle files.

In [82]:
# Load
with open("perm_boot_SCALES/ANBP_perm_scales.pkl", "rb") as f:
    ANBP_perm_scales = pickle.load(f)

with open("perm_boot_SCALES/ANR_perm_scales.pkl", "rb") as f:
    ANR_perm_scales = pickle.load(f)

with open("perm_boot_SCALES/BED_OSFED_perm_scales.pkl", "rb") as f:
    BED_OSFED_perm_scales = pickle.load(f)

with open("perm_boot_SCALES/BN_perm_scales.pkl", "rb") as f:
    BN_perm_scales = pickle.load(f)

# rebuild general dictionary
perm_diags_scales = {
    'ANBP' : ANBP_perm_scales,
    'ANR' : ANR_perm_scales,
    'BED_OSFED' : BED_OSFED_perm_scales,
    'BN' : BN_perm_scales
}

In [83]:
len(perm_diags_scales['ANBP'][5])

1668

In [86]:
base = 'SCALES_part1_plots/'
perm_summary_tables = {}
for diag, out in perm_diags_scales.items():
    perm_summary_tables[diag] = o_plots.save_all_figures(out, output_dir=base + diag + "_figures")

Saved -> SCALES_part1_plots/ANBP_figures\fig1_null_distributions.pdf
Saved -> SCALES_part1_plots/ANBP_figures\fig2_omega_scatter.pdf
Saved -> SCALES_part1_plots/ANBP_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


Saved -> SCALES_part1_plots/ANBP_figures\fig4_volcano.pdf
Supplementary table saved → SCALES_part1_plots/ANBP_figures\supplementary_table.csv  (3885 multiplets)
Saved -> SCALES_part1_plots/ANR_figures\fig1_null_distributions.pdf
Saved -> SCALES_part1_plots/ANR_figures\fig2_omega_scatter.pdf
Saved -> SCALES_part1_plots/ANR_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


Saved -> SCALES_part1_plots/ANR_figures\fig4_volcano.pdf
Supplementary table saved → SCALES_part1_plots/ANR_figures\supplementary_table.csv  (3806 multiplets)
Saved -> SCALES_part1_plots/BED_OSFED_figures\fig1_null_distributions.pdf
Saved -> SCALES_part1_plots/BED_OSFED_figures\fig2_omega_scatter.pdf
Saved -> SCALES_part1_plots/BED_OSFED_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


Saved -> SCALES_part1_plots/BED_OSFED_figures\fig4_volcano.pdf
Supplementary table saved → SCALES_part1_plots/BED_OSFED_figures\supplementary_table.csv  (3829 multiplets)
Saved -> SCALES_part1_plots/BN_figures\fig1_null_distributions.pdf
Saved -> SCALES_part1_plots/BN_figures\fig2_omega_scatter.pdf
Saved -> SCALES_part1_plots/BN_figures\fig3_omega_violin.pdf


C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8321 (\N{SUBSCRIPT ONE}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\utente\Documents\DataScience\TESI_MAGISTRALE\CODE\optim_help\plots\o_info_bootstrap_plots.py:602: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


Saved -> SCALES_part1_plots/BN_figures\fig4_volcano.pdf
Supplementary table saved → SCALES_part1_plots/BN_figures\supplementary_table.csv  (3842 multiplets)


#### Filter multiplets - pvalue wise + REDUNDANCY/SYNERGY check
- Filter out the multiplets whose observed $\Omega$ is not significant (pvalue based)
- Filter out the multiplets whose observed and significant $\Omega$ - information shows zero-interaction (set a tolerance for what is zero-interaction: |$\Omega$| < 0.05

In [90]:
omega_threshold = 0.10
#alpha_significance = 0.05

fdr_neut_filtered_mults_diags = {}
filtered_descriptive = {}
for diagnosis, perm_scales in perm_diags_scales.items():
    fdr_neut_filtered_mults_diags[diagnosis] = ut.fdr_neuter_filter(perm_scales, omega_threshold = omega_threshold)
    filtered_descriptive[diagnosis] = ut.fdr_neuter_filter(perm_scales, omega_threshold = 0)

In [92]:
omega_partitions_diags = {}
for diag, filtered_boot in filtered_descriptive.items():
    omega_partitions_diags[diag] = ut.omega_partition(filtered_boot, alpha = alpha_significance, omega_threshold = omega_threshold)
    
print(ut.describe_omega_partition(omega_partitions_diags))

Diagnosis: ANBP
	 Interation type: syn there are:
	 	 Order: 3 # multiplets: 111
	 	 Order: 4 # multiplets: 1346
	 	 Order: 5 # multiplets: 1545
	 Interation type: red there are:
	 	 Order: 3 # multiplets: 45
	 	 Order: 4 # multiplets: 41
	 	 Order: 5 # multiplets: 55
	 Interation type: neut there are:
	 	 Order: 3 # multiplets: 298
	 	 Order: 4 # multiplets: 90
	 	 Order: 5 # multiplets: 62
	 Interation type: non_sig there are:
	 	 Order: 3 # multiplets: 0
	 	 Order: 4 # multiplets: 0
	 	 Order: 5 # multiplets: 0
Diagnosis: ANR
	 Interation type: syn there are:
	 	 Order: 3 # multiplets: 9
	 	 Order: 4 # multiplets: 1151
	 	 Order: 5 # multiplets: 1527
	 Interation type: red there are:
	 	 Order: 3 # multiplets: 37
	 	 Order: 4 # multiplets: 34
	 	 Order: 5 # multiplets: 32
	 Interation type: neut there are:
	 	 Order: 3 # multiplets: 355
	 	 Order: 4 # multiplets: 99
	 	 Order: 5 # multiplets: 73
	 Interation type: non_sig there are:
	 	 Order: 3 # multiplets: 0
	 	 Order: 4 # multip

### BCa 

#### Bias Corrected permutation distribution

BCa preparation: get the sampling distribution (permute ROWS)


In [91]:
# here are the fildered multiplets to use for further computations
op_multiplets_diags_F = ut.get_multiplets_as_dict_list_F(fdr_neut_filtered_mults_diags)

In [93]:
bca_sampD_diags_inter = {}
for diagnosis, mults in op_multiplets_diags_F.items():
    start = time.perf_counter()
    bca_sampD_diags_inter[diagnosis] = bca_bootopt.get_boot_stats_parallel(multiplets_dict=mults, 
                                                                            X=EDI_diags[diagnosis], 
                                                                            n_boot=n_boot_perm)
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]
Multiplets | family 3:   0%|          | 0/156 [00:00<?, ?it/s]

  0%|          | 0/156 [00:00<?, ?it/s]

  1%|          | 1/156 [00:04<10:25,  4.04s/it]

  5%|▌         | 8/156 [00:04<00:56,  2.61it/s]

  8%|▊         | 12/156 [00:06<01:05,  2.21it/s]

  9%|▉         | 14/156 [00:06<00:51,  2.77it/s]

 10%|█         | 16/156 [00:06<00:41,  3.40it/s]

 12%|█▏        | 18/156 [00:08<01:02,  2.20it/s]

 13%|█▎        | 21/156 [00:08<00:46,  2.91it/s]

 15%|█▌        | 24/156 [00:09<00:31,  4.15it/s]

 17%|█▋        | 26/156 [00:10<00:47,  2.74it/s]

 18%|█▊        | 28/156 [00:10<00:37,  3.40it/s]

 19%|█▉        | 30/156 [00:11<00:33,  3.75it/s]

 20%|█▉        | 31/156 [00:11<00:30,  4.16it/s]

 21%|██        | 33/156 [00:12<00:45,  2.67it/s]

 23%|██▎       | 36/156 [00:13<00:34,  3.43it/s]

 24%|██▍       | 38/156 [00:13<00:27,  4.36it/s]

 25%|██▌       | 39/156 [00:13<00:27,  4.32it/s]

 26%|██▋       | 41/156 [00:14<00:38,  3.00it/s]


  5%|▍         | 69/1387 [00:24<06:02,  3.64it/s]

  5%|▌         | 70/1387 [00:24<06:19,  3.47it/s]

  5%|▌         | 71/1387 [00:25<05:53,  3.72it/s]

  5%|▌         | 73/1387 [00:26<10:24,  2.10it/s]

  5%|▌         | 74/1387 [00:26<08:34,  2.55it/s]

  5%|▌         | 75/1387 [00:27<08:25,  2.60it/s]

  6%|▌         | 77/1387 [00:27<06:12,  3.51it/s]

  6%|▌         | 78/1387 [00:27<05:41,  3.83it/s]

  6%|▌         | 79/1387 [00:27<05:05,  4.29it/s]

  6%|▌         | 80/1387 [00:27<04:21,  5.00it/s]

  6%|▌         | 81/1387 [00:29<12:08,  1.79it/s]

  6%|▌         | 82/1387 [00:29<10:44,  2.03it/s]

  6%|▌         | 84/1387 [00:30<07:48,  2.78it/s]

  6%|▌         | 86/1387 [00:30<05:30,  3.94it/s]

  6%|▋         | 87/1387 [00:30<06:12,  3.49it/s]

  6%|▋         | 89/1387 [00:32<09:40,  2.24it/s]

  6%|▋         | 90/1387 [00:32<08:54,  2.43it/s]

  7%|▋         | 92/1387 [00:32<07:05,  3.04it/s]

  7%|▋         | 93/1387 [00:32<06:23,  3.38it/s]

  7%|▋         | 94/1387 [00:33

 18%|█▊        | 252/1387 [01:31<08:28,  2.23it/s]

 18%|█▊        | 254/1387 [01:31<05:54,  3.20it/s]

 18%|█▊        | 255/1387 [01:31<05:43,  3.30it/s]

 18%|█▊        | 256/1387 [01:32<06:22,  2.95it/s]

 19%|█▊        | 257/1387 [01:32<05:42,  3.30it/s]

 19%|█▊        | 258/1387 [01:33<07:37,  2.47it/s]

 19%|█▊        | 260/1387 [01:34<09:15,  2.03it/s]

 19%|█▉        | 261/1387 [01:34<08:12,  2.29it/s]

 19%|█▉        | 262/1387 [01:34<06:52,  2.73it/s]

 19%|█▉        | 263/1387 [01:35<05:33,  3.37it/s]

 19%|█▉        | 264/1387 [01:35<05:58,  3.13it/s]

 19%|█▉        | 265/1387 [01:35<05:31,  3.39it/s]

 19%|█▉        | 266/1387 [01:36<09:05,  2.05it/s]

 19%|█▉        | 267/1387 [01:36<07:11,  2.59it/s]

 19%|█▉        | 268/1387 [01:37<08:51,  2.11it/s]

 19%|█▉        | 269/1387 [01:37<07:44,  2.41it/s]

 19%|█▉        | 270/1387 [01:38<07:54,  2.36it/s]

 20%|█▉        | 271/1387 [01:38<06:07,  3.04it/s]

 20%|█▉        | 272/1387 [01:38<05:06,  3.63it/s]

 20%|█▉     

 30%|██▉       | 414/1387 [02:29<04:56,  3.28it/s]

 30%|██▉       | 415/1387 [02:30<05:47,  2.80it/s]

 30%|██▉       | 416/1387 [02:30<05:08,  3.15it/s]

 30%|███       | 417/1387 [02:31<05:19,  3.04it/s]

 30%|███       | 418/1387 [02:31<06:12,  2.60it/s]

 30%|███       | 419/1387 [02:32<07:15,  2.22it/s]

 30%|███       | 421/1387 [02:32<05:05,  3.16it/s]

 30%|███       | 422/1387 [02:32<05:22,  2.99it/s]

 30%|███       | 423/1387 [02:33<05:51,  2.74it/s]

 31%|███       | 424/1387 [02:33<04:54,  3.27it/s]

 31%|███       | 425/1387 [02:33<04:41,  3.42it/s]

 31%|███       | 426/1387 [02:34<06:00,  2.67it/s]

 31%|███       | 427/1387 [02:34<06:57,  2.30it/s]

 31%|███       | 429/1387 [02:35<05:01,  3.18it/s]

 31%|███       | 430/1387 [02:35<05:19,  3.00it/s]

 31%|███       | 431/1387 [02:36<05:44,  2.77it/s]

 31%|███       | 432/1387 [02:36<06:12,  2.57it/s]

 31%|███▏      | 434/1387 [02:37<05:32,  2.87it/s]

 31%|███▏      | 435/1387 [02:37<06:00,  2.64it/s]

 31%|███▏   

 42%|████▏     | 578/1387 [03:30<05:07,  2.63it/s]

 42%|████▏     | 579/1387 [03:30<04:40,  2.88it/s]

 42%|████▏     | 580/1387 [03:30<05:21,  2.51it/s]

 42%|████▏     | 581/1387 [03:31<04:27,  3.02it/s]

 42%|████▏     | 582/1387 [03:31<04:29,  2.99it/s]

 42%|████▏     | 583/1387 [03:31<04:12,  3.19it/s]

 42%|████▏     | 584/1387 [03:32<05:00,  2.67it/s]

 42%|████▏     | 585/1387 [03:32<05:24,  2.47it/s]

 42%|████▏     | 586/1387 [03:32<04:49,  2.76it/s]

 42%|████▏     | 587/1387 [03:33<04:25,  3.01it/s]

 42%|████▏     | 588/1387 [03:33<04:56,  2.69it/s]

 42%|████▏     | 589/1387 [03:33<04:15,  3.12it/s]

 43%|████▎     | 590/1387 [03:34<05:07,  2.59it/s]

 43%|████▎     | 591/1387 [03:34<04:45,  2.79it/s]

 43%|████▎     | 592/1387 [03:34<04:38,  2.86it/s]

 43%|████▎     | 593/1387 [03:35<06:10,  2.14it/s]

 43%|████▎     | 594/1387 [03:35<04:50,  2.73it/s]

 43%|████▎     | 595/1387 [03:36<05:12,  2.54it/s]

 43%|████▎     | 596/1387 [03:36<05:27,  2.42it/s]

 43%|████▎  

...

 72%|███████▏  | 1163/1616 [13:01<06:36,  1.14it/s]




 72%|███████▏  | 1165/1616 [13:02<04:58,  1.51it/s]




 72%|███████▏  | 1166/1616 [13:03<04:17,  1.75it/s]




 72%|███████▏  | 1167/1616 [13:03<04:07,  1.82it/s]




 72%|███████▏  | 1168/1616 [13:04<04:31,  1.65it/s]




 72%|███████▏  | 1169/1616 [13:05<05:14,  1.42it/s]




 72%|███████▏  | 1170/1616 [13:06<05:57,  1.25it/s]




 72%|███████▏  | 1171/1616 [13:07<06:18,  1.18it/s]




 73%|███████▎  | 1173/1616 [13:07<04:32,  1.63it/s]




 73%|███████▎  | 1174/1616 [13:08<04:41,  1.57it/s]




 73%|███████▎  | 1175/1616 [13:08<03:58,  1.85it/s]




 73%|███████▎  | 1176/1616 [13:09<04:40,  1.57it/s]




 73%|███████▎  | 1177/1616 [13:10<05:20,  1.37it/s]




 73%|███████▎  | 1178/1616 [13:11<05:16,  1.39it/s]




 73%|███████▎  | 1179/1616 [13:12<05:58,  1.22it/s]




 73%|███████▎  | 1180/1616 [13:12<04:45,  1.53it/s]




 73%|███████▎  | 1181/1616 [13:13<04:38,  1.56it/s]




 73%|███████▎  | 1182/1616 [13:13<03:49,  1.89it

 80%|███████▉  | 1289/1616 [14:26<03:38,  1.50it/s]




 80%|███████▉  | 1290/1616 [14:26<03:14,  1.67it/s]




 80%|███████▉  | 1291/1616 [14:27<03:45,  1.44it/s]




 80%|███████▉  | 1292/1616 [14:29<05:00,  1.08it/s]




 80%|████████  | 1293/1616 [14:29<04:03,  1.33it/s]




 80%|████████  | 1294/1616 [14:29<03:22,  1.59it/s]




 80%|████████  | 1295/1616 [14:30<02:59,  1.79it/s]




 80%|████████  | 1296/1616 [14:31<03:12,  1.66it/s]




 80%|████████  | 1297/1616 [14:31<02:56,  1.80it/s]




 80%|████████  | 1298/1616 [14:32<03:09,  1.67it/s]




 80%|████████  | 1299/1616 [14:33<03:25,  1.54it/s]




 80%|████████  | 1300/1616 [14:34<04:38,  1.14it/s]




 81%|████████  | 1301/1616 [14:34<03:47,  1.38it/s]




 81%|████████  | 1302/1616 [14:35<03:04,  1.70it/s]




 81%|████████  | 1303/1616 [14:35<03:08,  1.66it/s]




 81%|████████  | 1304/1616 [14:36<03:33,  1.46it/s]




 81%|████████  | 1305/1616 [14:36<02:42,  1.91it/s]




 81%|████████  | 1306/1616 [14:37<02:51,  1.81it

 88%|████████▊ | 1414/1616 [15:51<02:26,  1.38it/s]




 88%|████████▊ | 1416/1616 [15:52<02:24,  1.39it/s]




 88%|████████▊ | 1418/1616 [15:54<02:32,  1.29it/s]




 88%|████████▊ | 1419/1616 [15:54<02:05,  1.56it/s]




 88%|████████▊ | 1420/1616 [15:54<01:56,  1.69it/s]




 88%|████████▊ | 1421/1616 [15:55<01:43,  1.89it/s]




 88%|████████▊ | 1422/1616 [15:56<02:26,  1.33it/s]




 88%|████████▊ | 1424/1616 [15:57<02:12,  1.45it/s]




 88%|████████▊ | 1426/1616 [15:59<02:38,  1.20it/s]




 88%|████████▊ | 1428/1616 [16:00<01:45,  1.78it/s]




 88%|████████▊ | 1429/1616 [16:00<01:34,  1.97it/s]




 88%|████████▊ | 1430/1616 [16:01<02:18,  1.34it/s]




 89%|████████▊ | 1431/1616 [16:02<01:52,  1.65it/s]




 89%|████████▊ | 1432/1616 [16:02<02:03,  1.49it/s]




 89%|████████▊ | 1433/1616 [16:03<02:05,  1.46it/s]




 89%|████████▊ | 1434/1616 [16:05<02:46,  1.10it/s]




 89%|████████▉ | 1436/1616 [16:05<01:51,  1.62it/s]




 89%|████████▉ | 1438/1616 [16:07<02:04,  1.43it

 96%|█████████▌| 1545/1616 [17:19<00:41,  1.70it/s]




 96%|█████████▌| 1546/1616 [17:19<00:44,  1.57it/s]




 96%|█████████▌| 1547/1616 [17:20<00:34,  2.02it/s]




 96%|█████████▌| 1548/1616 [17:21<00:43,  1.55it/s]




 96%|█████████▌| 1549/1616 [17:23<01:14,  1.11s/it]




 96%|█████████▌| 1550/1616 [17:23<00:55,  1.19it/s]




 96%|█████████▌| 1551/1616 [17:23<00:44,  1.47it/s]




 96%|█████████▌| 1552/1616 [17:23<00:34,  1.88it/s]




 96%|█████████▌| 1553/1616 [17:24<00:36,  1.72it/s]




 96%|█████████▌| 1554/1616 [17:25<00:36,  1.72it/s]




 96%|█████████▋| 1556/1616 [17:26<00:38,  1.57it/s]




 96%|█████████▋| 1557/1616 [17:28<00:56,  1.04it/s]




 96%|█████████▋| 1558/1616 [17:28<00:44,  1.31it/s]




 96%|█████████▋| 1559/1616 [17:28<00:33,  1.68it/s]




 97%|█████████▋| 1560/1616 [17:29<00:28,  1.98it/s]




 97%|█████████▋| 1561/1616 [17:29<00:28,  1.90it/s]




 97%|█████████▋| 1562/1616 [17:30<00:29,  1.83it/s]




 97%|█████████▋| 1563/1616 [17:30<00:23,  2.29it

Runtime: 1746.292645 s


In [95]:
# skip dir creation if it already exist
out_dir = "sampD_diags_SCALES"            
os.makedirs(out_dir, exist_ok=True)

for diagnosis, sampD_scales in bca_sampD_diags_inter.items():
    #fname = f"{diagnosis}_sampD_scales.pkl"
    fname = f"{diagnosis}_sampD_scales.pkl"
    fpath = os.path.join(out_dir, fname)
    with open(fpath, "wb") as f:
        pickle.dump(sampD_scales, f, protocol=pickle.HIGHEST_PROTOCOL)

In [43]:
#bca_sampD_diags_scales['ANR']['DT'][5]
#fdr_neut_filtered_diags_scales['ANR']['DT'][5]

In [97]:
# Load
with open("sampD_diags_SCALES/ANBP_sampD_scales.pkl", "rb") as f:
    ANBP_sampD_scales = pickle.load(f)

with open("sampD_diags_SCALES/ANR_sampD_scales.pkl", "rb") as f:
    ANR_sampD_scales = pickle.load(f)

with open("sampD_diags_SCALES/BED_OSFED_sampD_scales.pkl", "rb") as f:
    BED_OSFED_sampD_scales = pickle.load(f)

with open("sampD_diags_SCALES/BN_sampD_scales.pkl", "rb") as f:
    BN_sampD_scales = pickle.load(f)

# rebuild general dictionary
bca_sampD_diags_scales = {
    'ANBP' : ANBP_sampD_scales,
    'ANR' : ANR_sampD_scales,
    'BED_OSFED' : BED_OSFED_sampD_scales,
    'BN' : BN_sampD_scales
}

#### Assambling information:  
bca_sampD_diags_scales contains the emp distros from the bca sampling for each multiplet.
Assemble info from multiplets and sample distributions just computed, by adding to the dict of the fdr survived multiplets info about the BCa sample_distro

In [98]:
BCa_fdr_diags_scales = {}
for diagnosis in bca_sampD_diags_scales:
    BCa_fdr_diags_scales[diagnosis] = ut.get_BCa_input_dict_from_fdr(bca_sampD_diags_scales[diagnosis],
                                                              fdr_neut_filtered_mults_diags[diagnosis])

In [101]:
#BCa_fdr_diags_scales['ANR'][3][(38, 53, 64)]

In [102]:
bca_mults_diags_scales = {} 
for diagnosis, mults in BCa_fdr_diags_scales.items():
    start = time.perf_counter()
    # structure: order: multiple: samp_distro
    bca_mults_diags_scales[diagnosis] = bca_bootopt.bca_bootstrap_parallel(multiplets_dict=mults, 
                                                                            X=EDI_diags[diagnosis])
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]


[BCa | family 3] starting – 156 multiplets


Families:  33%|███▎      | 1/3 [00:02<00:04,  2.28s/it]

  [BCa | family 3] 156/156 multiplets (100%)

[BCa | family 4] starting – 1387 multiplets


Families:  67%|██████▋   | 2/3 [00:02<00:01,  1.31s/it]

...


[BCa | family 3] starting – 22 multiplets
  [BCa | family 3] 22/22 multiplets (100%)

[BCa | family 4] starting – 1305 multiplets


Families:  67%|██████▋   | 2/3 [00:00<00:00,  3.04it/s]

  [BCa | family 4] 1305/1305 multiplets (100%)

[BCa | family 5] starting – 1616 multiplets


Families: 100%|██████████| 3/3 [00:01<00:00,  2.02it/s]

  [BCa | family 5] 1616/1616 multiplets (100%)
Runtime: 1.484402 s


When running the BCa_CI_check procedure, the multiplets with not valid pvalue have already been filtered out.

In [103]:
from contextlib import redirect_stdout

BCa_selected_diags = {}
for diagnosis, multiplets in bca_mults_diags_scales.items():
    with open(diagnosis + "_CI_warnings_SCALES.txt", "w") as f:
        with redirect_stdout(f):
            BCa_selected_diags[diagnosis] = bca_bootopt.BCa_CI_mults_selection(multiplets)

In [106]:
BCa_selected_diags['ANR'][3][(38, 53, 64)]

'synergy'

Save results to picke files.

In [107]:
# skip dir creation if it already exist
out_dir_bca = "bca_selected_SCALES"            
os.makedirs(out_dir_bca, exist_ok=True)

for diagnosis, bca_scales in BCa_selected_diags.items():
    fname = f"{diagnosis}_bca_selected_scales.pkl"
    fpath = os.path.join(out_dir_bca, fname)
    with open(fpath, "wb") as f:
        pickle.dump(bca_scales, f, protocol=pickle.HIGHEST_PROTOCOL)

### Stage 2, part 2: BCa_CI: plots and stats

In [162]:
base = 'SCALES_part2_plots/'
n_boot = 1000

for diagnosis in bca_sampD_diags_scales:
    out_dir = f"{base}{diagnosis}_figures"
    
    # bootstrap sampling
    # boot_arrays = bootstrap_multiplets_chunked(multiplets_dict=multiplets, X=X, n_boot=500)
    boot_arrays = bca_sampD_diags_scales[diagnosis]

    # BCa confidence intervals
    # BCa_boot = bca_bootstrap_parallel(multiplets_dict=..., X=X, alpha=0.05)
    BCa_boot = bca_mults_diags_scales[diagnosis]
    
    # selection — keeps only {family: {multiplet: hypothesis}}
    # BCa_selected = BCa_CI_mults_selection(BCa_boot, tol=0.05)
    BCa_selected = BCa_selected_diags[diagnosis]
    
    # re-join with full info for the survivors  
    BCa_boot_sel, boot_arrays_sel = bca_bootopt.retrieve_selected_multiplets_info(
        BCa_selected = BCa_selected,
        BCa_boot_full = BCa_boot,
        boot_arrays_full = boot_arrays,
    )              # {family: {multiplet: (obs, lo, hi, diag)}}

    # pick representative multiplets and plot
    reps = bca_plots.pick_representative_multiplets(BCa_boot_sel, 
                                                    boot_arrays_sel)
    bca_plots.plot_all(
        BCa_boot_all = BCa_boot_sel,
        n_samples = EDI_diags[diagnosis].shape[0],
        n_boot = n_boot,
        representative_multiplets = reps,
        output_dir=out_dir,
    )

  Family 3: 104/156 multiplets retrieved.
  Family 4: 1357/1387 multiplets retrieved.
  Family 5: 219/1600 multiplets retrieved.

Representative multiplets selected from family 4:
  most_typical        multiplet=(65, 8, 60, 77)  Ω=-1.0906  CI=[-1.2647, -0.8217]  margin from 0=0.8217  method=percentile
  most_borderline     multiplet=(3, 6, 14, 22)  Ω=-0.3075  CI=[-0.4810, -0.0007]  margin from 0=0.0007  method=percentile
  most_extreme        multiplet=(54, 8, 40, 77)  Ω=-1.3667  CI=[-1.5397, -1.1053]  margin from 0=1.1053  method=percentile
Generating Figure 1 – Ω dot-and-whisker with BCa CIs ...
  saved → SCALES_part2_plots/ANBP_figures\fig1_omega_ci.pdf / .png
Generating Figure 2 – Redundancy / synergy breakdown ...
  saved → SCALES_part2_plots/ANBP_figures\fig2_breakdown.pdf / .png
Generating Figure 3 – Bootstrap distribution for multiplet (65, 8, 60, 77) ...
  saved → SCALES_part2_plots/ANBP_figures\fig3_boot_dist_fam4_mult65_8_60_77.pdf / .png
Generating Figure 3 – Bootstrap dist

### Hyperedges selection's results

- fdr_neut_filtered_diags_scales 

- bca_selected_diag

In [108]:
# Load
with open("bca_selected_SCALES/ANBP_bca_selected_scales.pkl", "rb") as f:
    ANBP_bca_sel_scales = pickle.load(f)

with open("bca_selected_SCALES/ANR_bca_selected_scales.pkl", "rb") as f:
    ANR_bca_sel_scales = pickle.load(f)

with open("bca_selected_SCALES/BED_OSFED_bca_selected_scales.pkl", "rb") as f:
    BED_OSFED_bca_sel_scales = pickle.load(f)

with open("bca_selected_SCALES/BN_bca_selected_scales.pkl", "rb") as f:
    BN_bca_sel_scales = pickle.load(f)

# rebuild general dictionary
BCa_selected_diags_loaded = {
    'ANBP' : ANBP_bca_sel_scales,
    'ANR' : ANR_bca_sel_scales,
    'BED_OSFED' : BED_OSFED_bca_sel_scales,
    'BN' : BN_bca_sel_scales
}

### For selected multiplets, recompute $\Omega$-information and append all the information together.

In [112]:
sel_mult_diags_scales = {}
for diagnosis, sel_mults in BCa_selected_diags_loaded.items():
    sel_mult_diags_scales[diagnosis] = pp_bootopt.compute_omega_for_multiplets(multiplets_dict=sel_mults, 
                                                                                   X=EDI_diags[diagnosis])

In [113]:
sel_mult_diags_scales['ANR'].head(5)

,order,label,omega,node1,node2,node3,node4,node5
0,3,redundancy,0.209966,2,45,32,<NA>,<NA>
1,3,synergy,-0.103594,3,22,39,<NA>,<NA>
2,3,redundancy,0.227318,7,16,32,<NA>,<NA>
3,3,redundancy,0.262536,7,16,49,<NA>,<NA>
4,3,redundancy,0.259711,7,32,49,<NA>,<NA>


In [114]:
# skip dir creation if it already exist
out_dir_sm = "selected_df_scales"            
os.makedirs(out_dir_sm, exist_ok=True)

for diagnosis, df in sel_mult_diags_scales.items():
    fname = f"{diagnosis}_selected_df_scales.csv"
    fpath = os.path.join(out_dir_sm, fname)
    df.to_csv(fpath, index=False)

### MULTIPLETS' CIs COMPARISON
I have:

    - BCa_selected_diags_scales_INTRA_loaded
    - BCa_selected_diags_INTER_loaded
    
I want to compare the bootstrap confidence interval of each multiplet with subsets of its components one order below.
Multiplets are discarded if the confidence intervals overlap, as this indicates that they do not display additional synergy or redundancy.

In [115]:
BCa_selected_diags_loaded['ANR'][3] # get all keys

{(38, 53, 64): 'synergy',
 (53, 61, 64): 'synergy',
 (9, 45, 55): 'redundancy',
 (9, 45, 59): 'redundancy',
 (9, 55, 59): 'redundancy',
 (9, 55, 62): 'redundancy',
 (9, 59, 62): 'redundancy',
 (19, 55, 62): 'redundancy',
 (31, 55, 62): 'redundancy',
 (45, 55, 59): 'redundancy',
 (45, 59, 62): 'redundancy',
 (55, 59, 62): 'redundancy',
 (7, 16, 32): 'redundancy',
 (7, 16, 49): 'redundancy',
 (7, 32, 49): 'redundancy',
 (16, 25, 32): 'redundancy',
 (16, 25, 49): 'redundancy',
 (16, 32, 49): 'redundancy',
 (70, 79, 90): 'synergy',
 (21, 26, 51): 'redundancy',
 (21, 26, 60): 'redundancy',
 (3, 22, 39): 'synergy',
 (13, 29, 52): 'synergy',
 (9, 45, 32): 'redundancy',
 (2, 45, 32): 'redundancy',
 (42, 43, 63): 'synergy'}

In [119]:
selected_as_list = ut.get_selected_INTER(BCa_selected_diags_loaded)

#### Get the distributions for CIs computation 

In [120]:
selected_boot_stats = {}
for diagnosis, mults in selected_as_list.items():
    start = time.perf_counter()
    selected_boot_stats[diagnosis] = bca_bootopt.get_boot_stats_parallel(multiplets_dict=mults, 
                                                                            X=EDI_diags[diagnosis], 
                                                                            n_boot=n_boot_perm) # 400
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]
Multiplets | family 3:   0%|          | 0/104 [00:00<?, ?it/s]

  0%|          | 0/104 [00:00<?, ?it/s]

  1%|          | 1/104 [00:04<07:14,  4.22s/it]

  3%|▎         | 3/104 [00:04<01:54,  1.13s/it]

  5%|▍         | 5/104 [00:04<00:57,  1.73it/s]

  7%|▋         | 7/104 [00:04<00:34,  2.81it/s]

  9%|▊         | 9/104 [00:06<00:51,  1.85it/s]

 11%|█         | 11/104 [00:06<00:36,  2.54it/s]

 12%|█▎        | 13/104 [00:06<00:28,  3.21it/s]

 15%|█▌        | 16/104 [00:06<00:17,  4.98it/s]

 17%|█▋        | 18/104 [00:08<00:30,  2.80it/s]

 18%|█▊        | 19/104 [00:08<00:31,  2.72it/s]

 21%|██        | 22/104 [00:09<00:19,  4.11it/s]

 22%|██▏       | 23/104 [00:09<00:18,  4.33it/s]

 24%|██▍       | 25/104 [00:10<00:28,  2.80it/s]

 26%|██▌       | 27/104 [00:10<00:24,  3.09it/s]

 27%|██▋       | 28/104 [00:11<00:22,  3.30it/s]

 28%|██▊       | 29/104 [00:11<00:20,  3.73it/s]

 29%|██▉       | 30/104 [00:11<00:18,  4.05it/s]

 3

  7%|▋         | 92/1357 [00:35<10:30,  2.01it/s]


  7%|▋         | 93/1357 [00:35<10:00,  2.10it/s]


  7%|▋         | 96/1357 [00:36<06:50,  3.08it/s]


  7%|▋         | 97/1357 [00:36<07:34,  2.77it/s]


  7%|▋         | 98/1357 [00:37<07:05,  2.96it/s]


  7%|▋         | 99/1357 [00:37<06:57,  3.01it/s]


  7%|▋         | 100/1357 [00:38<08:42,  2.40it/s]


  7%|▋         | 101/1357 [00:38<07:28,  2.80it/s]


  8%|▊         | 103/1357 [00:38<04:54,  4.27it/s]


  8%|▊         | 104/1357 [00:39<06:37,  3.15it/s]


  8%|▊         | 105/1357 [00:39<07:23,  2.82it/s]


  8%|▊         | 106/1357 [00:40<08:03,  2.59it/s]


  8%|▊         | 107/1357 [00:40<07:10,  2.91it/s]


  8%|▊         | 108/1357 [00:40<07:56,  2.62it/s]


  8%|▊         | 109/1357 [00:40<06:40,  3.11it/s]


  8%|▊         | 110/1357 [00:41<06:28,  3.21it/s]


  8%|▊         | 111/1357 [00:41<05:13,  3.98it/s]


  8%|▊         | 112/1357 [00:41<07:33,  2.74it/s]


  8%|▊         | 113/1357 [00:42<06:56,  2.98it/s]



 19%|█▊        | 254/1357 [01:30<07:16,  2.53it/s]


 19%|█▉        | 255/1357 [01:31<07:52,  2.33it/s]


 19%|█▉        | 256/1357 [01:31<06:28,  2.84it/s]


 19%|█▉        | 257/1357 [01:31<05:10,  3.54it/s]


 19%|█▉        | 258/1357 [01:31<06:28,  2.83it/s]


 19%|█▉        | 259/1357 [01:32<05:49,  3.14it/s]


 19%|█▉        | 260/1357 [01:32<07:58,  2.29it/s]


 19%|█▉        | 261/1357 [01:33<06:47,  2.69it/s]


 19%|█▉        | 262/1357 [01:33<05:43,  3.19it/s]


 19%|█▉        | 263/1357 [01:33<06:41,  2.72it/s]


 19%|█▉        | 264/1357 [01:34<08:13,  2.21it/s]


 20%|█▉        | 266/1357 [01:34<05:14,  3.47it/s]


 20%|█▉        | 267/1357 [01:34<05:04,  3.58it/s]


 20%|█▉        | 268/1357 [01:35<07:35,  2.39it/s]


 20%|█▉        | 270/1357 [01:36<06:12,  2.92it/s]


 20%|█▉        | 271/1357 [01:36<06:04,  2.98it/s]


 20%|██        | 272/1357 [01:37<07:38,  2.37it/s]


 20%|██        | 273/1357 [01:37<06:12,  2.91it/s]


 20%|██        | 274/1357 [01:37<05:54,  3.06i

 30%|███       | 410/1357 [02:24<04:35,  3.44it/s]


 30%|███       | 411/1357 [02:25<06:14,  2.52it/s]


 30%|███       | 412/1357 [02:25<07:05,  2.22it/s]


 31%|███       | 414/1357 [02:26<04:49,  3.26it/s]


 31%|███       | 415/1357 [02:26<05:42,  2.75it/s]


 31%|███       | 417/1357 [02:27<04:48,  3.26it/s]


 31%|███       | 418/1357 [02:27<04:36,  3.40it/s]


 31%|███       | 419/1357 [02:27<05:41,  2.75it/s]


 31%|███       | 420/1357 [02:28<06:41,  2.33it/s]


 31%|███       | 422/1357 [02:28<04:38,  3.36it/s]


 31%|███       | 423/1357 [02:29<05:06,  3.04it/s]


 31%|███       | 424/1357 [02:29<04:30,  3.45it/s]


 31%|███▏      | 425/1357 [02:29<04:39,  3.33it/s]


 31%|███▏      | 426/1357 [02:29<04:35,  3.38it/s]


 31%|███▏      | 427/1357 [02:30<05:50,  2.65it/s]


 32%|███▏      | 428/1357 [02:31<06:51,  2.26it/s]


 32%|███▏      | 430/1357 [02:31<04:26,  3.47it/s]


 32%|███▏      | 431/1357 [02:31<05:15,  2.93it/s]


 32%|███▏      | 432/1357 [02:32<04:39,  3.31i

 41%|████▏     | 563/1357 [03:18<04:02,  3.27it/s]


 42%|████▏     | 564/1357 [03:18<03:41,  3.59it/s]


 42%|████▏     | 565/1357 [03:18<03:58,  3.32it/s]


 42%|████▏     | 566/1357 [03:19<04:26,  2.97it/s]


 42%|████▏     | 567/1357 [03:19<05:14,  2.51it/s]


 42%|████▏     | 568/1357 [03:19<05:03,  2.60it/s]


 42%|████▏     | 569/1357 [03:20<05:15,  2.50it/s]


 42%|████▏     | 570/1357 [03:20<04:15,  3.08it/s]


 42%|████▏     | 571/1357 [03:20<03:23,  3.87it/s]


 42%|████▏     | 572/1357 [03:20<03:38,  3.60it/s]


 42%|████▏     | 573/1357 [03:21<03:46,  3.46it/s]


 42%|████▏     | 574/1357 [03:21<04:22,  2.98it/s]


 42%|████▏     | 575/1357 [03:22<04:52,  2.67it/s]


 42%|████▏     | 576/1357 [03:22<05:51,  2.22it/s]


 43%|████▎     | 577/1357 [03:23<05:26,  2.39it/s]


 43%|████▎     | 578/1357 [03:23<04:15,  3.04it/s]


 43%|████▎     | 579/1357 [03:23<04:24,  2.94it/s]


 43%|████▎     | 580/1357 [03:23<03:31,  3.67it/s]


 43%|████▎     | 581/1357 [03:23<03:32,  3.66i

...

 25%|██▍       | 116/468 [01:19<04:18,  1.36it/s]






 25%|██▌       | 117/468 [01:20<04:21,  1.34it/s]






 25%|██▌       | 118/468 [01:20<03:31,  1.65it/s]






 25%|██▌       | 119/468 [01:20<03:17,  1.77it/s]






 26%|██▌       | 120/468 [01:21<03:48,  1.53it/s]






 26%|██▌       | 121/468 [01:22<03:40,  1.58it/s]






 26%|██▌       | 122/468 [01:22<03:09,  1.82it/s]






 26%|██▋       | 123/468 [01:23<03:18,  1.74it/s]






 26%|██▋       | 124/468 [01:24<04:06,  1.39it/s]






 27%|██▋       | 125/468 [01:25<04:17,  1.33it/s]






 27%|██▋       | 126/468 [01:25<03:18,  1.72it/s]






 27%|██▋       | 127/468 [01:25<03:05,  1.84it/s]






 27%|██▋       | 128/468 [01:27<04:38,  1.22it/s]






 28%|██▊       | 130/468 [01:28<03:30,  1.60it/s]






 28%|██▊       | 131/468 [01:28<02:49,  1.99it/s]






 28%|██▊       | 132/468 [01:29<03:35,  1.56it/s]






 28%|██▊       | 133/468 [01:30<04:10,  1.34it/s]






 29%|██▊       | 134/468 [01:31<04:17,  1.30it/s

 50%|████▉     | 233/468 [02:35<02:00,  1.95it/s]






 50%|█████     | 234/468 [02:35<02:28,  1.57it/s]






 50%|█████     | 235/468 [02:36<02:38,  1.47it/s]






 50%|█████     | 236/468 [02:37<02:41,  1.44it/s]






 51%|█████     | 237/468 [02:38<02:31,  1.53it/s]






 51%|█████     | 238/468 [02:38<02:04,  1.85it/s]






 51%|█████     | 239/468 [02:40<03:25,  1.12it/s]






 51%|█████▏    | 240/468 [02:40<02:49,  1.35it/s]






 52%|█████▏    | 242/468 [02:41<02:23,  1.57it/s]






 52%|█████▏    | 243/468 [02:42<02:31,  1.48it/s]






 52%|█████▏    | 244/468 [02:43<02:39,  1.40it/s]






 52%|█████▏    | 245/468 [02:43<02:06,  1.76it/s]






 53%|█████▎    | 246/468 [02:43<01:54,  1.94it/s]






 53%|█████▎    | 247/468 [02:45<03:07,  1.18it/s]






 53%|█████▎    | 248/468 [02:45<02:22,  1.55it/s]






 53%|█████▎    | 249/468 [02:45<01:48,  2.02it/s]






 53%|█████▎    | 250/468 [02:46<02:24,  1.51it/s]






 54%|█████▎    | 251/468 [02:47<02:56,  1.23it/s

 74%|███████▍  | 347/468 [03:50<00:58,  2.08it/s]






 74%|███████▍  | 348/468 [03:51<01:08,  1.76it/s]






 75%|███████▍  | 349/468 [03:52<01:35,  1.24it/s]






 75%|███████▍  | 350/468 [03:53<01:30,  1.30it/s]






 75%|███████▌  | 351/468 [03:54<01:40,  1.16it/s]






 75%|███████▌  | 352/468 [03:54<01:17,  1.50it/s]






 75%|███████▌  | 353/468 [03:55<01:01,  1.88it/s]






 76%|███████▌  | 354/468 [03:56<01:14,  1.52it/s]






 76%|███████▌  | 355/468 [03:56<01:07,  1.67it/s]






 76%|███████▌  | 356/468 [03:56<00:58,  1.92it/s]






 76%|███████▋  | 357/468 [03:58<01:40,  1.10it/s]






 76%|███████▋  | 358/468 [03:59<01:21,  1.35it/s]






 77%|███████▋  | 359/468 [04:00<01:32,  1.18it/s]






 77%|███████▋  | 361/468 [04:00<00:56,  1.89it/s]






 77%|███████▋  | 362/468 [04:01<00:58,  1.82it/s]






 78%|███████▊  | 363/468 [04:01<01:03,  1.66it/s]






 78%|███████▊  | 364/468 [04:02<00:56,  1.83it/s]






 78%|███████▊  | 365/468 [04:04<01:34,  1.09it/s

 99%|█████████▊| 461/468 [05:07<00:05,  1.31it/s]






 99%|█████████▊| 462/468 [05:08<00:03,  1.53it/s]






 99%|█████████▉| 463/468 [05:09<00:03,  1.46it/s]






 99%|█████████▉| 464/468 [05:09<00:02,  1.90it/s]






 99%|█████████▉| 465/468 [05:09<00:01,  1.78it/s]






100%|█████████▉| 466/468 [05:10<00:01,  1.65it/s]






100%|█████████▉| 467/468 [05:10<00:00,  1.95it/s]






Families: 100%|██████████| 3/3 [11:41<00:00, 233.99s/it]

Runtime: 701.968459 s


In [127]:
# skip dir creation if it already exist
out_dir_sm = "selected_boot_stat_SCALES"            
os.makedirs(out_dir_sm, exist_ok=True)

for diagnosis, stat_scales in selected_boot_stats.items():
    fname = f"{diagnosis}_selected_stat_scales.pkl"
    fpath = os.path.join(out_dir_sm, fname)
    with open(fpath, "wb") as f:
        pickle.dump(stat_scales, f, protocol=pickle.HIGHEST_PROTOCOL)

In [128]:
# LOAD
with open("selected_boot_stat_SCALES/ANBP_selected_stat_scales.pkl", "rb") as f:
    ANBP_sel_stat_scales = pickle.load(f)

with open("selected_boot_stat_SCALES/ANR_selected_stat_scales.pkl", "rb") as f:
    ANR_sel_stat_scales = pickle.load(f)

with open("selected_boot_stat_SCALES/BED_OSFED_selected_stat_scales.pkl", "rb") as f:
    BED_OSFED_sel_stat_scales = pickle.load(f)

with open("selected_boot_stat_SCALES/BN_selected_stat_scales.pkl", "rb") as f:
    BN_sel_stat_scales = pickle.load(f)

# rebuild general dictionary
selected_boot_stats_loaded = {
    'ANBP' : ANBP_sel_stat_scales,
    'ANR' : ANR_sel_stat_scales,
    'BED_OSFED' : BED_OSFED_sel_stat_scales,
    'BN' : BN_sel_stat_scales
}

Add observed info to dict for computing BCa CIs.

In [130]:
for diagnosis, diag_data in selected_boot_stats.items():
    for order, mults in diag_data.items():
        for mult, distro in mults.items():
            selected_boot_stats[diagnosis][order][mult] = {
                'observed_O' : o_infoptim.o_information(EDI_diags[diagnosis][list(mult)]),
                'samp_distro' : distro
            }

#### BCA

In [131]:
bca_final = {}
for diagnosis, mult_scales in selected_boot_stats.items():
    start = time.perf_counter()
    # structure: order: multiple: samp_distro
    bca_final[diagnosis] = bca_bootopt.bca_bootstrap_parallel(multiplets_dict=mult_scales, 
                                                                            X=EDI_diags[diagnosis])
    end = time.perf_counter()
    print(f"Runtime: {end - start:.6f} s")

Families:   0%|          | 0/3 [00:00<?, ?it/s]


[BCa | family 3] starting – 104 multiplets


Families:  33%|███▎      | 1/3 [00:02<00:04,  2.31s/it]

  [BCa | family 3] 104/104 multiplets (100%)

[BCa | family 4] starting – 1357 multiplets


Families: 100%|██████████| 3/3 [00:03<00:00,  1.02s/it]


...


[BCa | family 3] starting – 17 multiplets
  [BCa | family 3] 17/17 multiplets (100%)

[BCa | family 4] starting – 767 multiplets


Families:  67%|██████▋   | 2/3 [00:00<00:00,  5.12it/s]

  [BCa | family 4] 767/767 multiplets (100%)

[BCa | family 5] starting – 468 multiplets


Families: 100%|██████████| 3/3 [00:00<00:00,  4.62it/s]

  [BCa | family 5] 468/468 multiplets (100%)
Runtime: 0.653508 s


In [140]:
bca_final['BN'][3][(4, 5, 38)]

(0.1422196931366706,
 0.004647885587052604,
 0.1887394982661235,
 {'method': 'percentile', 'skew': 0.07344770227253146})

#### Actually filtering out the overlapping multiplets

In [146]:
filtered_diags = {} 
discarded_diags = {}
for diagnose, orders in bca_final.items():
    filtered, discarded = sel.select_multiplets_minimal(orders)
    filtered_diags[diagnose] = filtered
    discarded_diags[diagnose] = discarded

In [151]:
filtered_diags['BN'][3]

{(2, 12, 19): '',
 (4, 5, 38): '',
 (7, 25, 68): '',
 (9, 45, 55): '',
 (9, 45, 59): '',
 (9, 45, 62): '',
 (9, 55, 59): '',
 (9, 55, 62): '',
 (21, 51, 60): '',
 (22, 66, 68): '',
 (28, 47, 53): '',
 (31, 55, 62): '',
 (45, 55, 59): '',
 (45, 55, 62): '',
 (45, 59, 62): '',
 (55, 59, 62): '',
 (72, 79, 90): ''}

### FINAL STEP ( $\Omega$ - DF)

In [152]:
sel_finale = {}
for diagnosis, sel_mults in filtered_diags.items():
    sel_finale[diagnosis] = pp_bootopt.compute_omega_for_multiplets(multiplets_dict=sel_mults, 
                                                                                   X=EDI_diags[diagnosis])
    ap = sel_finale[diagnosis]
    sel_finale[diagnosis]["label"] = np.where(ap["omega"] < 0, "synergy", "redundancy")

In [156]:
sel_finale['ANR'].head()

,order,label,omega,node1,node2,node3,node4,node5
0,3,redundancy,0.209966,2,32,45,<NA>,<NA>
1,3,synergy,-0.103594,3,22,39,<NA>,<NA>
2,3,redundancy,0.227318,7,16,32,<NA>,<NA>
3,3,redundancy,0.262536,7,16,49,<NA>,<NA>
4,3,redundancy,0.259711,7,32,49,<NA>,<NA>


In [165]:
print('Redundancy count:', (sel_finale['ANBP']["label"] == 'redundancy').sum())
print('Synergy count:', (sel_finale['ANBP']["label"] == 'synergy').sum())
print(len(sel_finale['ANBP']))

print('Redundancy count:', (sel_finale['ANR']["label"] == 'redundancy').sum())
print('Synergy count:', (sel_finale['ANR']["label"] == 'synergy').sum())
print(len(sel_finale['ANR']))

print('Redundancy count:', (sel_finale['BED_OSFED']["label"] == 'redundancy').sum())
print('Synergy count:', (sel_finale['BED_OSFED']["label"] == 'synergy').sum())
print(len(sel_finale['BED_OSFED']))

print('Redundancy count:', (sel_finale['BN']["label"] == 'redundancy').sum())
print('Synergy count:', (sel_finale['BN']["label"] == 'synergy').sum())
print(len(sel_finale['BN']))

Redundancy count: 26
Synergy count: 1353
1379
Redundancy count: 23
Synergy count: 1149
1172
Redundancy count: 55
Synergy count: 1191
1246
Redundancy count: 19
Synergy count: 855
874


In [159]:
# skip dir creation if it already exist
out_dir_sm = "selected_df_scales_FINAL"            
os.makedirs(out_dir_sm, exist_ok=True)

for diagnosis, df in sel_finale.items():
    fname = f"{diagnosis}_selected_df_scales.csv"
    fpath = os.path.join(out_dir_sm, fname)
    df.to_csv(fpath, index=False)